# 004 — Sommerfeld-Norton FE three-term Ez, Stollberg

ITU Handbook on Ground Wave Propagation (2014) Part 1 §3.2.1. Mirrors
ITU sheet columns BQ through CY for row 7.

This is the **flat-earth (FE) model**. There is no curvature, no
horizon, no shadow zone. The field at the receiver is the coherent sum
of three contributions:

1. direct ray $E_d$ along the geometric line $r_1$,
2. ground-reflected ray $E_r$ along $r_2$ with Fresnel reflection coefficient $R_v$,
3. Norton surface wave $E_s$ with attenuation function $F$.

$$E_z = E_d + E_r + E_s$$

The cross beam (Stollberg) runs from Stollberg over sea (sigma = 5 S/m, eps_r = 70) to a
6 000 m He 111 receiver. Change `target` in the next cell to any
name from `TARGETS` and re-run.


In [1]:
import math, cmath, pandas as pd
from IPython.display import Markdown, display
import common as c

station = 'Stollberg'
target  = 'Beeston'      # change me

s = c.STATIONS[station]
t = c.TARGETS[target]
f_MHz = s['freq_MHz']
ground = 'sea' if target.startswith('TF') else s['ground']
sigma = c.GROUND[ground]['sigma']
eps_r = c.GROUND[ground]['eps_r']


## Geometry on a flat earth

Straight-line slant ranges to the receiver (no curvature):

$$r_1 = \sqrt{d^2 + (h_{rx} - h_{tx})^2}, \qquad r_2 = \sqrt{d^2 + (h_{rx} + h_{tx})^2}$$

Grazing angles $\psi_1$, $\psi_2$ measured from the surface:

$$\cos^2\psi_1 = (d/r_1)^2, \quad \cos^2\psi_2 = (d/r_2)^2, \quad \sin\psi_2 = (h_{rx} + h_{tx})/r_2$$

The receiver image is the ground reflection point of the source.


In [2]:
d_m  = c.great_circle_m(s['lat_deg'], s['lon_deg'], t['lat'], t['lon'])
h_tx = s['h_tx_m']
h_rx = t['rx_alt_m']
lam  = c.freq_to_wavelen(f_MHz)
k    = c.wavenumber(f_MHz)
r1 = math.sqrt(d_m**2 + (h_rx - h_tx)**2)
r2 = math.sqrt(d_m**2 + (h_rx + h_tx)**2)
cos2_psi1 = (d_m/r1)**2
cos2_psi2 = (d_m/r2)**2
sin_psi2  = (h_rx+h_tx)/r2
display(Markdown(f'''
- d = {d_m/1000:.2f} km, lambda = {lam:.4f} m, k = {k:.4f} rad/m
- r1 = {r1:.2f} m, r2 = {r2:.2f} m
- cos^2 psi1 = {cos2_psi1:.6f}, cos^2 psi2 = {cos2_psi2:.6f}
- sin psi2 = {sin_psi2:.6f}
'''))



- d = 693.50 km, lambda = 9.5172 m, k = 0.6602 rad/m
- r1 = 693522.00 m, r2 = 693523.24 m
- cos^2 psi1 = 0.999927, cos^2 psi2 = 0.999923
- sin psi2 = 0.008755


## Complex ground impedance

$$x = \frac{18\,000\,\sigma}{f_{MHz}}, \qquad n^2 = \varepsilon_r - j\,x, \qquad u^2 = \frac{2}{n^2}$$

Fresnel reflection coefficient (vertical polarisation):

$$R_v = \frac{n^2 \sin\psi_2 - \sqrt{n^2 - \cos^2\psi_2}}{n^2 \sin\psi_2 + \sqrt{n^2 - \cos^2\psi_2}}$$


In [3]:
x  = 18000.0*sigma/f_MHz
n2 = complex(eps_r, -x)
u2 = 2.0/n2
sqrt_term = cmath.sqrt(n2 - cos2_psi2)
Rv = (n2*sin_psi2 - sqrt_term) / (n2*sin_psi2 + sqrt_term)
display(Markdown(f'''
- x = 18000 sigma / f = {x:.4f}
- n^2 = {n2}
- u^2 = {u2}
- sqrt(n^2 - cos^2 psi2) = {sqrt_term}
- R_v = {Rv}
'''))



- x = 18000 sigma / f = 2857.1429
- n^2 = (70-2857.1428571428573j)
- u^2 = (1.713971188793926e-05+0.0006995800770587454j)
- sqrt(n^2 - cos^2 psi2) = (38.255561616597184-37.34284292801083j)
- R_v = (-0.4133601796705901-0.3460027519858561j)


## Numerical distance $w$ and attenuation function $F$

Norton's numerical distance for vertical polarisation:

$$w = \frac{-j\,2\,k\,r_2\,u^2\,(1 - u^2 \cos^2\psi_2)}{1 - R_v}$$

For large $|w|$ the attenuation function is well approximated by its
asymptotic expansion (ITU Handbook Part 1 §3.2.1 Eq. 7):

$$F \approx -\frac{1}{2w} - \frac{3}{(2w)^2} - \frac{15}{(2w)^3} - \frac{105}{(2w)^4}$$

This is exactly the formula used by ITU sheet column CJ.


In [4]:
one_minus_Rv = 1 - Rv
one_minus_u2_cos2 = 1 - u2*cos2_psi2
minus_j_2kr2 = complex(0.0, -2*k*r2)
w_num = minus_j_2kr2 * u2 * one_minus_u2_cos2
w = w_num / one_minus_Rv
two_w = 2*w
F = (-1/two_w) + (-3/two_w**2) + (-15/two_w**3) + (-105/two_w**4)
display(Markdown(f'''
- w = {w}
- |w| = {abs(w):.4g}
- F = {F}
- |F| = {abs(F):.4g}
'''))



- w = (424.97677460025324-115.45966920019987j)
- |w| = 440.4
- F = (-0.0010990123612137147-0.0002996471101301306j)
- |F| = 0.001139


## Three-term superposition

Each term has its own propagator $\exp(-j k r)$ and aperture factor:

$$E_d = \frac{\cos^2\psi_1}{r_1} e^{-j k r_1}$$
$$E_r = \frac{\cos^2\psi_2}{r_2}\, R_v\, e^{-j k r_2}$$
$$E_s = \frac{(1 - R_v)(1 - u^2 + u^4 \cos^2\psi_2)\,F}{r_2} e^{-j k r_2}$$

Total complex field (normalised per unit source strength):

$$E_z = E_d + E_r + E_s$$


In [5]:
exp_jkr1 = cmath.exp(complex(0, -k*r1))
exp_jkr2 = cmath.exp(complex(0, -k*r2))
direct  = cos2_psi1 * exp_jkr1 / r1
reflect = cos2_psi2 * Rv * exp_jkr2 / r2
u4 = u2**2
bracket = 1 - u2 + u4*cos2_psi2
surface = one_minus_Rv * bracket * F * exp_jkr2 / r2
Ez_sum = direct + reflect + surface
display(Markdown(f'''
- E_direct  = {direct}
- E_reflect = {reflect}
- E_surface = {surface}
- E_z sum   = {Ez_sum}
- |E_z|     = {abs(Ez_sum):.6g}
'''))



- E_direct  = (2.5313546272778914e-07-1.419414827198487e-06j)
- E_reflect = (-3.956998553359878e-08+7.762084165784963e-07j)
- E_surface = (3.3451082191316386e-10+2.3664659142045733e-09j)
- E_z sum   = (2.138999880161035e-07-6.408399447057861e-07j)
- |E_z|     = 6.75595e-07


## Absolute field strength and link budget

The ITU sheet normalises to $\sqrt{90 P_{tx}}$ for the short-dipole
reference, then boosts by the 99x29 m aperture directivity (less the
1.5 baseline of a half-wave dipole):

$$|E_z|_{abs} = \sqrt{90\,P_{tx}} \cdot |E_z|_{sum}$$
$$E_{boost} = |E_z|_{abs} \cdot \sqrt{G_{tx,lin} / 1.5}$$

Received power at an isotropic receive antenna:

$$P_{rx} = \frac{E_{boost}^2 \lambda^2}{8\pi \eta_0}, \qquad \eta_0 = 376.73\ \Omega$$


In [6]:
sqrt_90P = math.sqrt(90 * s['Ptx_W'])
G_tx_dBi = c.aperture_gain_dBi(s['W_m'], s['H_m'], f_MHz)
G_tx_lin = 10**(G_tx_dBi/10)
Ez_Vpm   = sqrt_90P * abs(Ez_sum)
E_boost  = Ez_Vpm * math.sqrt(G_tx_lin / 1.5)
P_rx_W   = E_boost**2 * lam**2 / (8*math.pi*c.ETA_0)
P_rx_dBW = 10*math.log10(P_rx_W)
display(Markdown(f'''
- sqrt(90 P_tx) = {sqrt_90P:.3f}
- G_tx = {G_tx_dBi:.2f} dBi   (linear {G_tx_lin:.2f})
- |E_z| absolute = {Ez_Vpm:.6g} V/m
- E_boost        = {E_boost:.6g} V/m
- P_rx           = {P_rx_W:.4g} W
- P_rx           = **{P_rx_dBW:.3f} dBW**
'''))



- sqrt(90 P_tx) = 519.615
- G_tx = 26.00 dBi   (linear 398.31)
- |E_z| absolute = 0.00035105 V/m
- E_boost        = 0.0057205 V/m
- P_rx           = 3.131e-07 W
- P_rx           = **-65.044 dBW**


## Equisignal SNR and verdict

Crossover and noise floor follow the same formulas as the Fock
chapter. The flat-earth model has no diffraction loss, so the
equisignal SNR scales only with FSPL plus the squint crossover.


In [7]:
r = c.link_budget(station, target, model='sommerfeld')
verdict = ('PASS' if r['SNR_eq_dB'] >= 10
           else 'MARGINAL' if r['SNR_eq_dB'] >= 0
           else 'FAIL')
pd.DataFrame({
    'Quantity':[
        'P_rx (dBW)',
        'Noise floor (dBW)',
        'SNR peak (dB)',
        'Crossover at 5 deg (dB)',
        'SNR equisignal (dB)',
        'V_eq at 50 ohm (uV)',
        'V_noise at 50 ohm (uV)',
        'Verdict',
    ],
    'Value':[
        r['P_rx_dBW'], r['N_dBW'], r['SNR_peak_dB'],
        r['crossover_dB'], r['SNR_eq_dB'],
        r['V_eq_uV'], r['V_noise_uV'], verdict,
    ],
})


,Quantity,Value
0,P_rx (dBW),-65.043816
1,Noise floor (dBW),-159.44663
2,SNR peak (dB),94.402814
3,Crossover at 5 deg (dB),-19.86749
4,SNR equisignal (dB),74.535324
5,V_eq at 50 ohm (uV),402.193198
6,V_noise at 50 ohm (uV),0.075452
7,Verdict,PASS


## Sweep all confirmed Stollberg paths


In [8]:
rows = []
for tgt in ['Beeston', 'Derby', 'Birmingham', 'Liverpool', 'TF 400 km', 'TF 500 km', 'TF 700 km', 'TF 800 km', 'TF 1000 km']:
    rr = c.link_budget(station, tgt, model='sommerfeld')
    v = ('PASS' if rr['SNR_eq_dB'] >= 10
         else 'MARGINAL' if rr['SNR_eq_dB'] >= 0
         else 'FAIL')
    rows.append((tgt, round(rr['d_km'],1), rr['ground'],
                 round(rr['P_rx_dBW'],2),
                 round(rr['SNR_peak_dB'],2),
                 round(rr['SNR_eq_dB'],2),
                 round(rr['V_eq_uV'],3), v))
pd.DataFrame(rows, columns=['target','d_km','ground','P_rx_dBW',
                            'SNRpeak_dB','SNReq_dB','V_eq_uV','verdict'])


,target,d_km,ground,P_rx_dBW,SNRpeak_dB,SNReq_dB,V_eq_uV,verdict
0,Beeston,693.5,sea,-65.04,94.40,74.54,402.193,PASS
1,Derby,710.1,sea,-65.37,94.08,74.21,387.451,PASS
2,Birmingham,753.8,sea,-66.20,93.25,73.38,352.129,PASS
3,Liverpool,790.6,sea,-66.87,92.57,72.71,325.841,PASS
4,TF 400 km,413.7,sea,-60.03,99.42,79.55,716.636,PASS
5,TF 500 km,504.7,sea,-62.73,96.71,76.85,524.794,PASS
6,TF 700 km,702.6,sea,-67.56,91.89,72.02,301.150,PASS
7,TF 800 km,805.3,sea,-69.64,89.81,69.94,236.999,PASS
8,TF 1000 km,1000.1,sea,-73.02,86.43,66.56,160.530,PASS
